In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv('UNSTRUCTURED_API_KEY')
if api_key:
	os.environ["UNSTRUCTURED_API_KEY"] = api_key
else:
	print("Warning: UNSTRUCTURED_API_KEY environment variable not set")

In [3]:
import pickle
from pathlib import Path

cache_path = Path("../data/processed/docs_cache.pkl")

# Load from cache if exists
if cache_path.exists():
    with open(cache_path, 'rb') as f:
        docs = pickle.load(f)
    print("Loaded from cache")
else:
    # Load via API
    from langchain_unstructured import UnstructuredLoader
    loader = UnstructuredLoader(
        file_path="../data/raw/finetunexlmr.pdf",
        api_key=api_key,
        partition_via_api=True,
    )
    docs = loader.load()
    
    # Save to cache
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    with open(cache_path, 'wb') as f:
        pickle.dump(docs, f)
    print("Loaded from API and cached")

Loaded from cache


In [4]:
import pickle
from pathlib import Path

# Save loaded documents
cache_path = Path("../data/processed/docs_cache.pkl")
cache_path.parent.mkdir(parents=True, exist_ok=True)

with open(cache_path, 'wb') as f:
    pickle.dump(docs, f)

print(f"Documents saved to {cache_path}")

Documents saved to ../data/processed/docs_cache.pkl


In [5]:
docs[4].page_content

'This research proposes a three-stage pipeline:'

In [6]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv('GROQ_API_KEY'),
    temperature=0
)

# response = llm.invoke("Explain RAG in one sentence.")
# print(response.content)


/home/bigyan/Desktop/mero_pdf/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

In [8]:
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

# Check current document structure
print(f"Original documents: {len(docs)}")
if docs:
    print(f"Sample doc length: {len(docs[0].page_content)} chars")

# Apply semantic chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", "? ", "! ", "; ", " ", ""],
    length_function=len,
)

chunked_docs = text_splitter.split_documents(docs)
print(f"After chunking: {len(chunked_docs)} chunks")

Original documents: 64
Sample doc length: 122 chars
After chunking: 64 chunks


In [9]:
system_prompt = """Instructions:
1.Use the provided context to answer the user's query
2. If the context contains related information, use it to answer as best as you can
3. Only say you don't have information if the context is completely unrelated
"""

In [10]:
from langchain_community.vectorstores import Qdrant

# Create vectorstore and add documents
vectorstore = Qdrant.from_documents(
    documents=chunked_docs,
    embedding=model,
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY"),
    collection_name="pdf_chunks",
    force_recreate=True,  # Recreate if exists
)

print(f"Added {len(chunked_docs)} documents to Qdrant!")

Added 64 documents to Qdrant!


In [11]:
retriever = vectorstore.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k":2}
)

In [12]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "Context:{context} Input:{input}")
])

In [13]:
from langchain_classic.chains import create_history_aware_retriever

retriever_prompt = ChatPromptTemplate.from_messages([
    ("system","Rewrite the user query using the chat history if needed. Do NOT answer."),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

In [14]:
history_aware_retriever = create_history_aware_retriever(
    llm,retriever,retriever_prompt
)

In [15]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
question_answer_chain = create_stuff_documents_chain(
    llm,prompt=answer_prompt
)

In [16]:
from langchain_core.runnables import RunnableParallel
rag_chain = (
    RunnableParallel({
        'input':lambda d: d['input'],
        'chat_history':lambda d: d['chat_history'],
        'context': lambda d:history_aware_retriever.invoke({
            'chat_history': d['chat_history'],
            'input': d['input'],
        })
    })
    | question_answer_chain
)

In [17]:
query = 'What is the research about?'

search_results = retriever.invoke(query)


In [18]:
chat_history=[]

In [19]:
from langchain_core.messages import AIMessage,HumanMessage
response = rag_chain.invoke({
    'input':query,
    'chat_history':chat_history
})
chat_history.extend(
    [
        HumanMessage(content=query),
        AIMessage(content=response)
    ]
)

In [20]:
response

'Based on the provided context, it appears to be related to a research methodology or experimental design. However, the specific details about the research topic are not mentioned.\n\nTo answer your question, I would say that I don\'t have enough information to determine what the research is about. The context only mentions "Experimental Design Input" and "What is the research about?" without providing any further details.'